# 02 | Churn target construction (10-day horizon)

Definition of prediction cutoffs, target verification, and four prediction windows.

**Source:** `archive/Project_fin_original.ipynb`. The original computations, structure, and results have been preserved. The notebook text has been translated into English; standalone setup and local checkpoints support reproducible execution.


In [ ]:
# Standalone setup: no src/ folder required.
# Run the notebooks in numerical order (02 -> 07); 01 is optional exploration.
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

working_dir = Path.cwd().resolve()
# Works when VS Code/Jupyter starts in the repository root or notebooks/.
if working_dir.name == "notebooks":
    project_root = working_dir.parent
elif (working_dir / "notebooks").is_dir():
    project_root = working_dir
elif (working_dir.parent / "notebooks").is_dir():
    project_root = working_dir.parent
else:
    project_root = working_dir

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

HORIZON = pd.Timedelta(days=10)
CUTOFFS = pd.date_range(start="2018-10-11", end="2018-11-10", freq="10D")
FEATURES_12 = [
    "n_sessions_10d", "n_active_days_10d", "n_songs_10d",
    "n_likes_10d", "n_dislikes_10d", "n_errors_10d",
    "n_playlist_adds_10d", "total_song_length_10d", "n_artists_10d",
    "downgraded_10d", "is_paid", "songs_trend_10d"
]
EVENT_COLS = [
    "userId", "time", "registration", "page", "sessionId", "level",
    "gender", "length", "artist", "song", "status", "method",
    "userAgent", "auth", "firstName", "lastName", "itemInSession", "location"
]

def raw_path(kind):
    if kind not in ("train", "test"):
        raise ValueError("kind must be 'train' or 'test'")
    filename = f"{kind}.parquet"
    candidates = [
        project_root / filename,
        project_root / "data" / "raw" / filename,
        project_root / "notebooks" / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Missing {filename}. Put it in the repository root or data/raw/. "
        f"Current root: {project_root}"
    )

def load_events(kind):
    frame = pd.read_parquet(raw_path(kind), columns=EVENT_COLS)
    frame["userId"] = frame["userId"].astype(str)
    frame["time"] = pd.to_datetime(frame["time"])
    frame["registration"] = pd.to_datetime(frame["registration"])
    return frame

def data_path(filename):
    folder = project_root / "data" / "processed"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def artifact_path(filename):
    folder = project_root / "artifacts"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def save_cv_state(dev_data, holdout_data, cv_splits):
    dev_data.to_parquet(data_path("dev_data.parquet"), index=False)
    holdout_data.to_parquet(data_path("holdout_data.parquet"), index=False)
    fold_arrays = {}
    for i, (idx_train, idx_valid) in enumerate(cv_splits):
        fold_arrays[f"train_{i}"] = np.asarray(idx_train, dtype=np.int64)
        fold_arrays[f"valid_{i}"] = np.asarray(idx_valid, dtype=np.int64)
    np.savez_compressed(data_path("cv_splits.npz"), **fold_arrays)

def load_cv_state():
    dev = pd.read_parquet(data_path("dev_data.parquet"))
    holdout = pd.read_parquet(data_path("holdout_data.parquet"))
    with np.load(data_path("cv_splits.npz")) as saved:
        n = sum(name.startswith("train_") for name in saved.files)
        splits = [
            (saved[f"train_{i}"].copy(), saved[f"valid_{i}"].copy())
            for i in range(n)
        ]
    return dev, holdout, splits

print("Project folder:", project_root)

train = load_events("train")


# 2) Create the churn target

In [16]:
# Prediction horizon: 10 days
HORIZON = pd.Timedelta(days=10)

# Dates at which we simulate a prediction
CUTOFFS = pd.date_range(
    start="2018-10-11",
    end="2018-11-10",
    freq="10D"
)

# Display prediction windows
for cutoff in CUTOFFS:
    target_end = cutoff + HORIZON

    print(f"Prediction date: {cutoff.date()}")
    print(f"Churn observed between: {cutoff.date()} and {target_end.date()} (end excluded)")
    print("-" * 50)

Prediction date: 2018-10-11
Churn observed between: 2018-10-11 et 2018-10-21 (end excluded)
--------------------------------------------------
Prediction date: 2018-10-21
Churn observed between: 2018-10-21 et 2018-10-31 (end excluded)
--------------------------------------------------
Prediction date: 2018-10-31
Churn observed between: 2018-10-31 et 2018-11-10 (end excluded)
--------------------------------------------------
Prediction date: 2018-11-10
Churn observed between: 2018-11-10 et 2018-11-20 (end excluded)
--------------------------------------------------


In [17]:
# First prediction date
cutoff = CUTOFFS[0]
target_end = cutoff + HORIZON

# Cancellation date for each user
cancellations = (
    train[train["page"] == "Cancellation Confirmation"]
    .groupby("userId")["time"]
    .min()
)

# Users observed before the prediction date
users = pd.Index(
    train.loc[train["time"] < cutoff, "userId"].unique()
)

# Exclude users who have already cancelled
already_churned = cancellations[
    cancellations < cutoff
].index

users = users[~users.isin(already_churned)]

# Cancellations in the next 10 days
future_churn = cancellations[
    (cancellations >= cutoff) &
    (cancellations < target_end)
].index

# Create the target variable
labels = pd.DataFrame({"userId": users})

labels["churn"] = (
    labels["userId"].isin(future_churn).astype(int)
)

# Sanity checks
print("Number of users:", len(labels))

print("\nChurn distribution:")
print(labels["churn"].value_counts())

print("\nChurn rate:")
print(round(labels["churn"].mean() * 100, 2), "%")

display(labels.head())


Number of users: 14176

Churn distribution:
churn
0    13355
1      821
Name: count, dtype: int64

Churn rate:
5.79 %


,userId,churn
0,1749042,0
1,1563081,0
2,1697168,0
3,1222580,0
4,1714398,0


In [18]:
# List storing target labels for each period
all_labels = []

for cutoff in CUTOFFS:

    target_end = cutoff + HORIZON

    # Users observed before the prediction date
    users = pd.Index(
        train.loc[train["time"] < cutoff, "userId"].unique()
    )

    # Exclude users who have already cancelled
    already_churned = cancellations[
        cancellations < cutoff
    ].index

    users = users[~users.isin(already_churned)]

    # Users who cancel in the next 10 days
    future_churn = cancellations[
        (cancellations >= cutoff) &
        (cancellations < target_end)
    ].index

    # Create target labels for this window
    labels_period = pd.DataFrame({"userId": users})

    labels_period["cutoff"] = cutoff

    labels_period["churn"] = (
        labels_period["userId"]
        .isin(future_churn)
        .astype(int)
    )

    # Append this window's labels to the list
    all_labels.append(labels_period)

# Concatenate all windows
labels_all = pd.concat(all_labels, ignore_index=True)

# Summary by prediction date
summary = labels_all.groupby("cutoff").agg(
    n_users=("churn", "size"),
    n_churn=("churn", "sum"),
    churn_rate=("churn", "mean")
)

summary["churn_rate"] *= 100

display(summary)

all_labels


,n_users,n_churn,churn_rate
cutoff,,,
2018-10-11,14176,821,5.791479
2018-10-21,15452,817,5.287341
2018-10-31,15530,789,5.080489
2018-11-10,15276,663,4.340141


[        userId     cutoff  churn
 0      1749042 2018-10-11      0
 1      1563081 2018-10-11      0
 2      1697168 2018-10-11      0
 3      1222580 2018-10-11      0
 4      1714398 2018-10-11      0
 ...        ...        ...    ...
 14171  1208557 2018-10-11      0
 14172  1896647 2018-10-11      0
 14173  1494890 2018-10-11      0
 14174  1729584 2018-10-11      0
 14175  1878214 2018-10-11      0
 
 [14176 rows x 3 columns],
         userId     cutoff  churn
 0      1749042 2018-10-21      1
 1      1563081 2018-10-21      0
 2      1697168 2018-10-21      0
 3      1222580 2018-10-21      1
 4      1714398 2018-10-21      0
 ...        ...        ...    ...
 15447  1777544 2018-10-21      0
 15448  1668254 2018-10-21      0
 15449  1527915 2018-10-21      0
 15450  1510873 2018-10-21      0
 15451  1391436 2018-10-21      0
 
 [15452 rows x 3 columns],
         userId     cutoff  churn
 0      1563081 2018-10-31      0
 1      1697168 2018-10-31      0
 2      1714398 2018-10-

In [ ]:
# Save target labels for the next notebook
labels_all.to_parquet(data_path("labels_all.parquet"), index=False)
print("Target labels saved:", data_path("labels_all.parquet"))